In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
p_path = os.path.join(path,'Q1_data.csv')
df = pd.read_csv(p_path)

print(f"Shape: {df.shape}")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('delivery_time')
plt.show()

In [ ]:
# Task 1: Write your code here:
df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
# Missing values
# Define stat columns
stat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs','Delivery_Time']

# Drop rows with missing stat values
df_clean = df.dropna(subset=stat_cols).copy()
df_clean.fillna('none')
print(f"Shape after cleaning: {df_clean.shape}")
print("Missing values:")
print(df_clean.isnull().sum())

In [ ]:
# Task 3: Write your code here:
print(df_clean.duplicated().sum())
df.drop_duplicates(inplace=True)
print(df.duplicated().sum())

In [ ]:
feature_cols=pd.DataFrame(['Weather'	,'Traffic_Level'	,'Time_of_Day'	,'Vehicle_Type','Preparation_Time_min','Distance_km','Preparation_Time_min','Courier_Experience_yrs'])

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder

catg=pd.DataFrame(['Weather'	,'Traffic_Level'	,'Time_of_Day'	,'Vehicle_Type','Preparation_Time_min'])
onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
data_onehot_encoded = onehot_encoder.fit_transform(catg) # Apply fit_transform to the copied

print('\nData after encoding:\n', data_onehot_encoded) #show after encoding

In [ ]:
# Task 5: Write your code here:
# Define features (X) and target (y)
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Task 6: Write your code here:
import seaborn as sns

def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True))
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(df, "Delivery_Time")


In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1)
y = df["Delivery_Time"]


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error
model = RandomForestClassifier(n_estimators=100, max_depth=15,
                               class_weight='balanced', random_state=42)
model.fit(X, y)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  {mae:,.2f}")

for train_idx, val_idx in kfold.split(X):
    X_fold_train, X_fold_val =X_train_scaled[train_idx], X[val_idx]
    y_fold_train, y_fold_val =y_train.iloc[train_idx], y.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))


mae_scores = np.array(mae_scores)

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('delivery_time')
plt.show()

In [ ]:
# Task Bonus: Write your code here: